# Open Table Formats — Delta vs Iceberg vs Hudi> **Engineering Crash Courses** · [Web verzió](./index.html) · [Vissza a főoldalra](../index.html)Ez egy futtatható **Jupyter notebook** formátum, párhuzamosan a web-alapú kurzussal.Itt ugyanazokat a kódrészleteket tudod lokálisan, saját környezetben végigcsinálni.## Hogyan futtasd```bash# 1. Virtuális környezet (Python 3.10+)python -m venv .venv# Windows:.venv\Scripts\activate# macOS/Linux:source .venv/bin/activate# 2. Telepítsd a függőségeket (a notebook első cellája)# 3. Indítsd a Jupytertjupyter lab# vagyjupyter notebook```Minden cella saját magában értelmezhető. A `# %%` kommentek Jupytekben és VSCode-ban is a cellák határát jelölik.

## 1. Környezet — deltalake + pyiceberg

In [ ]:
%pip install deltalake pyarrow pandas 'pyiceberg[pyarrow]' --quiet

## 2. Delta Lake — egyszerű write

In [ ]:
import pandas as pdfrom deltalake import DeltaTable, write_deltalakefrom pathlib import PathPath('formats').mkdir(exist_ok=True)df = pd.DataFrame({    'order_id': [1, 2, 3, 4, 5],    'customer': ['A', 'B', 'A', 'C', 'B'],    'amount':   [1000, 2500, 750, 4200, 1800],})write_deltalake('formats/delta_orders', df, mode='overwrite')print('Delta table létrehozva')# Olvasás + historydt = DeltaTable('formats/delta_orders')print('Verzió:', dt.version())print('\nFájlok:', dt.files())

## 3. Iceberg — SQL catalog + writeAz Iceberg saját metadata struktúrával rendelkezik. pyiceberg Python-ból tudjuk közvetlenül vezérelni.

In [ ]:
from pyiceberg.catalog import load_catalogfrom pyiceberg.schema import Schemafrom pyiceberg.types import IntegerType, LongType, StringType, DoubleType, NestedFieldimport pyarrow as pa# SQLite-alapú catalog (fejlesztéshez)catalog = load_catalog(    'local',    **{        'type': 'sql',        'uri':  'sqlite:///formats/iceberg_catalog.db',        'warehouse': 'file://' + str(Path('formats/iceberg_warehouse').absolute()),    })Path('formats/iceberg_warehouse').mkdir(exist_ok=True)catalog.create_namespace_if_not_exists('webshop')schema = Schema(    NestedField(1, 'order_id', LongType(),   required=True),    NestedField(2, 'customer', StringType(), required=True),    NestedField(3, 'amount',   DoubleType(), required=True),)# Delete-recreate a demo kedvéérttry:    catalog.drop_table('webshop.orders')except Exception:    passtbl = catalog.create_table('webshop.orders', schema=schema)tbl.append(pa.Table.from_pandas(df, preserve_index=False))print('Iceberg table létrehozva')print('\nSnapshot-ok száma:', len(tbl.snapshots()))

## 4. Összehasonlítás — mely formátum mikor?

In [ ]:
import pandas as pdcomparison = pd.DataFrame([    {'feature': 'ACID tranzakciók',      'Delta': '✓', 'Iceberg': '✓', 'Hudi': '✓'},    {'feature': 'Time travel',           'Delta': '✓', 'Iceberg': '✓', 'Hudi': '✓'},    {'feature': 'Schema evolution',      'Delta': '✓', 'Iceberg': '✓', 'Hudi': '✓'},    {'feature': 'Partition evolution',   'Delta': '○', 'Iceberg': '✓', 'Hudi': '○'},    {'feature': 'Hidden partitioning',   'Delta': '○', 'Iceberg': '✓', 'Hudi': '○'},    {'feature': 'MERGE (upsert)',        'Delta': '✓', 'Iceberg': '✓', 'Hudi': '✓ (natív)'},    {'feature': 'Spark támogatás',       'Delta': '✓', 'Iceberg': '✓', 'Hudi': '✓'},    {'feature': 'Flink támogatás',       'Delta': '○', 'Iceberg': '✓', 'Hudi': '✓'},    {'feature': 'Databricks natív',      'Delta': '✓', 'Iceberg': '○', 'Hudi': '○'},    {'feature': 'AWS Glue catalog',      'Delta': '○', 'Iceberg': '✓', 'Hudi': '✓'},])comparison

## 5. Döntési útmutató- **Delta Lake** — ha Databricks / Spark natív stack-et építesz.- **Iceberg** — ha multi-engine (Spark + Flink + Trino + Snowflake), AWS-centric, vagy jövőálló open standard kell.- **Hudi** — ha streaming upsert a fő use-case (CDC, akár 2 perces latency).**Interoperabilitás:** Delta UniForm + XTable → ugyanazt a fájlt több formátumban olvashatod.

## Következő lépések- Térj vissza a [web-alapú kurzushoz](open-table-formats/index.html) a teljes anyagért, diagramokért és kvízekért.- Kapcsolódó források és videók a kurzusoldal alján találhatók a "További tanulás" szekcióban.- Ha elakadsz: [GitHub Issues](https://github.com/lugosidomotor/engineering_crash_courses/issues)---*Engineering Crash Courses · MIT licenc · Magyar Data & AI Engineering kurzusok*